[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pachterlab/OP_2025_2/blob/main/analysis_scripts/Design_Choices.ipynb)

# Design choices and sensitivity analyses (Supp Fig S4, Supp Table S5)

This notebook justifies accumap's settings and tests how its accuracy depends on its assumptions:
1. the **density-prior weight** λ and **fitting the kernel once** on the UMAP layout (**Supp Fig S4**);
2. the **number of passes** (one second stage) and of **refinement rounds** (Supp Table S5g), chosen by an imaging-free rule: stop when the anisotropy of contact directions reaches the level expected from sampling alone (Supplementary Note 2);
3. robustness to **non-uniform bead deposition** (**Supp Table S5f**);
4. the earlier sensitivity analyses of the single-stage method (**Supp Table S5a-e**, shown from cached results).

Design choices were made on development data only: replicate c58_8 and the calibrated simulation. The other three replicates are used only for validation (`Experimental_Ground_Truth.ipynb`).

**Runtime.** By default (`RECOMPUTE = False`) every result is drawn from `paper_results/` in about a minute. `RECOMPUTE = True` reruns sections 1-3 (about 6 hours on one GPU) into `paper_results/design_recomputed/`. Recomputation needs the c58_8 data from [SCP2577](https://singlecell.broadinstitute.org/single_cell/study/SCP2577/computational-reconstruction-for-imaging-free-spatial-transcriptomics) in the repository's `data/` folder.

In [ ]:
# Colab only: clone the repository (main branch) and install accumap; skipped when run locally
import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("/content/OP_2025_2"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main", "https://github.com/pachterlab/OP_2025_2.git", "/content/OP_2025_2"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/OP_2025_2"], check=True)
if "google.colab" in sys.modules:
    os.chdir("/content/OP_2025_2/analysis_scripts")

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.spatial import distance_matrix
from accumap.metrics import evaluate
from accumap.real_data import load_hu
from accumap.realistic_sim import simulate_realistic, load_params

RECOMPUTE = False                      # True: rerun sections 1-3 (GPU, about 6 h)
RESULTS = "paper_results"
OUTDIR = os.path.join(RESULTS, "design_recomputed") if RECOMPUTE else RESULTS
FIGURES = "../final_figures"
os.makedirs(OUTDIR, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
if RECOMPUTE:
    import torch
    assert torch.cuda.is_available(), "RECOMPUTE needs a CUDA GPU"
    from accumap.count_model import reconstruct, fit_kernel_at, contact_anisotropy

parse = lambda m: [float(x) for x in re.findall(r"(?:p50|p90|pairRMSE|J@10) ([\d.]+)", m)]
fmt = lambda m: f"recv p50 {m['median']:.1f} p90 {m['p90']:.1f} pairRMSE {m['pairwise_rmse']:.1f} J@10 {m['jaccard10']:.3f}"


def umap_start(rep, seed):
    f = "c58_8_joint_proto_umap.npz" if (rep, seed) == ("c58_8", 0) else f"{rep}_umapinit_seed{seed}.npz"
    return np.load(os.path.join(RESULTS, "umap_starts", f))["X0"]


def umap_embed(C, seed):
    import umap
    E = umap.UMAP(random_state=seed, metric="cosine", n_neighbors=25, min_dist=0.99, n_components=2,
                  n_epochs=400).fit_transform(np.log1p(C))
    E = E - E.mean(0)
    return E * 3000.0 / np.max(distance_matrix(E, E))

## 1. Density-prior weight and kernel (Supp Fig S4)

On c58_8, from 8 UMAP starts, with the full method:
- **A.** The density-prior weight λ in the warp objective. λ = 0 keeps the count likelihood and the second stage's isotropy prior; *prior only* switches the likelihood off.
- **B.** Refitting the transfer kernel on the method's own output and rerunning makes neighborhood recovery worse, and so does the kernel fit to imaging: the broader kernel fit once on the UMAP layout acts as a regularizer against the Poisson likelihood's overconfidence (Supplementary Note 2).

In [ ]:
if RECOMPUTE:
    C, X_true, U_true = load_hu("c58_8", count="umi")
    rows = []
    def rec(seed, kind, value, X):
        rows.append({"seed": seed, "kind": kind, "value": value, "metrics": fmt(evaluate(X, X_true))}); print(rows[-1], flush=True)
    for s in range(8):
        E = umap_start("c58_8", s)
        for lam in (0.0, 30.0, 300.0, 3000.0):
            R = reconstruct(C, E, 3000.0, lam=lam, verbose=False); rec(s, "lambda", lam, R["receivers"])
            if lam == 300.0:
                R0 = R
        rec(s, "prior_only", None, reconstruct(C, E, 3000.0, phi=1e30, isotropy=None, verbose=False)["receivers_warp"])
        rec(s, "refits", 0, R0["receivers"]); R = R0
        for k in (1, 2, 3):
            R = reconstruct(C, E, 3000.0, kernel=fit_kernel_at(C, R["receivers"], R["senders"], 3000.0), verbose=False)
            rec(s, "refits", k, R["receivers"])
        rec(s, "imaging_kernel", None, reconstruct(C, E, 3000.0, kernel=fit_kernel_at(C, X_true, U_true, 3000.0), verbose=False)["receivers"])
    with open(os.path.join(OUTDIR, "s4_full.jsonl"), "w") as fh:
        fh.writelines(json.dumps(r) + "\n" for r in rows)

In [ ]:
import sys
TITLE_FS, LABEL_FS, TICK_FS, LETTER_FS = 13, 12, 10, 18
C_UMAP, C_METHOD, C_ALT, C_REF = "#dc143c", "#4169e1", "#1baf7a", "#8a8986"
parse = lambda m: [float(x) for x in re.findall(r"(?:p50|p90|pairRMSE|J@10) ([\d.]+)", m)]
R = [json.loads(l) for l in open(os.path.join(OUTDIR, "s4_full.jsonl"))]
umap = {json.loads(l)["seed"]: parse(json.loads(l)["umap"]) for l in open(os.path.join(RESULTS, "hu_v2.jsonl"))
        if json.loads(l)["rep"] == "c58_8"}
U = np.array([umap[s] for s in sorted(umap)])


def get(kind, value, i):
    return np.array([parse(r["metrics"])[i] for r in R if r["kind"] == kind and r["value"] == value])


def style(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=TICK_FS)


def points(ax, x, y, color, marker="o"):
    jit = np.random.default_rng(0).uniform(-0.08, 0.08, len(y))
    ax.scatter(x + jit, y, s=16, color=color, alpha=0.45, linewidths=0, marker=marker, zorder=2)
    ax.plot([x - 0.18, x + 0.18], [y.mean()] * 2, color=color, lw=2.4, zorder=3)


fig, axes = plt.subplots(1, 2, figsize=(11, 4.9))
ax = axes[0]
lams = (0.0, 30.0, 300.0, 3000.0)
for k, lam in enumerate(lams):
    points(ax, k, get("lambda", lam, 0), C_METHOD)
ax.plot(range(len(lams)), [get("lambda", l, 0).mean() for l in lams], color=C_METHOD, lw=1, zorder=1)
points(ax, len(lams) + 0.2, get("prior_only", None, 0), C_REF, "s")
ax.axhline(U[:, 0].mean(), color=C_UMAP, ls="--", lw=1.8)
ax.text(len(lams) + 0.6, U[:, 0].mean() + 2, "UMAP", color=C_UMAP, fontsize=TICK_FS + 1, ha="right", fontweight="bold")
ax.set_xticks(list(range(len(lams))) + [len(lams) + 0.2])
ax.set_xticklabels(["0\n(counts\nonly)", "30", "300", "3000", "prior\nonly"], fontsize=TICK_FS)
ax.set_xlabel("Density-prior weight $\\lambda$", fontsize=LABEL_FS)
ax.set_ylabel("Median per-bead error ($\\mu$m)", fontsize=LABEL_FS)
ax.set_xlim(-0.5, len(lams) + 0.7)
top = max(get("prior_only", None, 0).max(), U[:, 0].max()) * 1.65
ax.set_ylim(0, top)
ax.text(0.03, 0.97, "warp objective:\n"
        r"$\frac{1}{\phi}\,\ell(X,U)\;-\;\lambda\sum_{h}\sum_{g}\left(\log\rho_h(g)-\overline{\log\rho_h}\right)^2$"
        "\n" r"$\ell$: count log-likelihood;  $\rho_h$: bead density",
        transform=ax.transAxes, ha="left", va="top", fontsize=TICK_FS + 1, linespacing=1.6,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#c9c8c4", lw=0.8))
ax.set_title("c58_8: density prior", fontsize=TITLE_FS)
style(ax); ax.text(-0.18, 1.04, "A", transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold")

ax = axes[1]
for k in range(4):
    points(ax, k, get("refits", k, 3), C_METHOD)
ax.plot(range(4), [get("refits", k, 3).mean() for k in range(4)], color=C_METHOD, lw=1, zorder=1)
points(ax, 4.2, get("imaging_kernel", None, 3), C_ALT, "D")
ax.axhline(U[:, 3].mean(), color=C_UMAP, ls="--", lw=1.8)
ax.text(4.6, U[:, 3].mean() + 0.003, "UMAP", color=C_UMAP, fontsize=TICK_FS + 1, ha="right", va="bottom", fontweight="bold")
ax.set_xticks([0, 1, 2, 3, 4.2]); ax.set_xticklabels(["0", "1", "2", "3", "imaging-\nfit kernel"], fontsize=TICK_FS)
ax.set_xlabel("Kernel refits on reconstruction", fontsize=LABEL_FS)
ax.set_ylabel("10-NN Jaccard distance  (← better)", fontsize=LABEL_FS)
ax.set_xlim(-0.5, 4.7)
ax.set_title("c58_8: kernel", fontsize=TITLE_FS)
style(ax); ax.text(-0.18, 1.04, "B", transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "FigS4_final.png"), dpi=300, bbox_inches="tight")
plt.show()

## 2. Number of passes

After each pass, the anisotropy of contact directions (the quantity the isotropy prior penalizes) is compared with the level expected from finite sampling alone, computed by replacing every contact direction with a random one. At the true positions this ratio is about 1. One second stage brings reconstructions to or below that level, so further passes have no excess anisotropy left to remove; accumap therefore runs one second stage. `reconstruct(..., verbose=True)` prints this ratio after each pass, so the rule can be checked on any dataset.

In [ ]:
if RECOMPUTE:
    from accumap.metrics import evaluate as ev
    out = []
    jobs = [("c58_8", s) for s in range(8)] + [("realistic", s) for s in range(5)]
    for ds, s in jobs:
        if ds == "c58_8":
            C, X_true, _ = load_hu("c58_8", count="umi"); E = umap_start("c58_8", s)
        else:
            C, X_true, _ = simulate_realistic(seed=s); E = umap_embed(C, s)
        R = reconstruct(C, E, 3000.0, n_passes=5, log_passes=True, verbose=False)
        out.append({"dataset": ds, "seed": s, "passes": [{"pass": k + 1, "penalty": p["penalty"], "floor": p["floor"],
                     "metrics": fmt(ev(p["receivers"], X_true))} for k, p in enumerate(R["pass_log"])]})
    with open(os.path.join(OUTDIR, "noise_floor.jsonl"), "w") as fh:
        fh.writelines(json.dumps(r) + "\n" for r in out)
    truth = {}
    C, X_true, U_true = load_hu("c58_8", count="umi"); truth["c58_8"] = contact_anisotropy(C, X_true, U_true, 3000.0)
    for s in range(5):
        C, X_true, U_true = simulate_realistic(seed=s); truth[f"realistic_{s}"] = contact_anisotropy(C, X_true, U_true, 3000.0)
    json.dump(truth, open(os.path.join(OUTDIR, "floor_at_truth.json"), "w"), indent=1)

runs = [json.loads(l) for l in open(os.path.join(OUTDIR, "noise_floor.jsonl"))]
truth = json.load(open(os.path.join(OUTDIR, "floor_at_truth.json")))
rows = []
for ds, label in (("realistic", "calibrated simulation"), ("c58_8", "c58_8")):
    G = [r for r in runs if r["dataset"] == ds]
    for k in range(len(G[0]["passes"])):
        P = [g["passes"][k] for g in G]
        rows.append({"data": label, "pass": k + 1, "anisotropy / floor": np.mean([p["penalty"] / p["floor"] for p in P]),
                     "median error (um)": np.mean([parse(p["metrics"])[0] for p in P]),
                     "J@10": np.mean([parse(p["metrics"])[3] for p in P])})
passes = pd.DataFrame(rows)
at_truth = {"c58_8": truth["c58_8"]["ratio"], "calibrated simulation": np.mean([truth[f"realistic_{s}"]["ratio"] for s in range(5)])}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for label, col in (("calibrated simulation", "#eda100"), ("c58_8", "#4169e1")):
    D = passes[passes["data"] == label]
    axes[0].plot(D["pass"], D["anisotropy / floor"], "o-", color=col, label=label)
    axes[0].axhline(at_truth[label], color=col, ls=":", lw=1.2)
    axes[1].plot(D["pass"], D["median error (um)"], "o-", color=col, label=label)
axes[0].set_ylabel("contact anisotropy / sampling floor\n(dotted: at the true positions)"); axes[1].set_ylabel("median error (um)")
for ax in axes:
    ax.set_xlabel("pass (1 = first stage)"); ax.set_xticks(range(1, 6)); ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
plt.tight_layout(); plt.show()
passes.round(3)

## Refinement rounds (Supp Table S5g)

Each stage ends with per-bead refinement: every receiver, then every sender, moves to the most likely position on a grid around its current one. Two rounds per stage (the first version of the method) are compared with four rounds and with the default: up to 12 rounds per stage, stopping early once fewer than 1 percent of beads move. c58_8 and the calibrated simulation chose the setting; c58_4, c58_5 and c58_16 validate it. These runs used the implementation before the speedups (cached results; `RECOMPUTE` does not rerun them).

The second table shows how often the 1 percent criterion ends a stage before the 12-round cap.

In [ ]:
rr = [json.loads(l) for l in open(os.path.join(RESULTS, "refine_rounds.jsonl"))]
LABEL = {"r2": "2 rounds per stage", "r4": "4 rounds per stage", "conv": "up to 12 (default)"}
rows, stops = [], []
for ds in ("c58_8", "c58_4", "c58_5", "c58_16", "realistic"):
    base = {r["seed"]: parse(r["metrics"]) for r in rr if r["dataset"] == ds and r["variant"] == "r2"}
    for v in ("r2", "r4", "conv"):
        G = [r for r in rr if r["dataset"] == ds and r["variant"] == v]
        M = np.array([parse(g["metrics"]) for g in G]); B = np.array([base[g["seed"]] for g in G])
        rows.append({"data": ds, "setting": LABEL[v], "n": len(G), "median (um)": M[:, 0].mean(), "p90 (um)": M[:, 1].mean(),
                     "pairwise RMSE (um)": M[:, 2].mean(), "J@10": M[:, 3].mean(),
                     "J@10 better than 2 rounds": f"{int((M[:, 3] < B[:, 3]).sum())}/{len(G)}" if v != "r2" else ""})
        if v == "conv":
            for g in G:          # split the round log into stages (each stage restarts at the +-15 window)
                st, cur = [], []
                for e in g["refine_log"]:
                    if e["window"] == 15.0 and cur:
                        st.append(cur); cur = []
                    cur.append(e)
                st.append(cur)
                for k, s in enumerate(st):
                    stops.append({"data": ds, "stage": k + 1, "rounds": len(s), "stopped early": s[-1]["moved_frac"] < 0.01})
display(pd.DataFrame(rows).set_index(["data", "setting"]).round(3))
S = pd.DataFrame(stops)
S.groupby(["stage"]).agg(runs=("rounds", "size"), mean_rounds=("rounds", "mean"), stopped_early=("stopped early", "sum")).round(1)

### Past the 12-round cap

The same refinement with the cap raised to 40 rounds per stage (c58_8 starts 0-2, calibrated arrays 0-1; metrics recorded after every round). Most stages converge given more rounds, but accuracy levels off well before then, and the final results agree with the 12-round runs (Supp Table S4 / Table S2 caches) to within run-to-run variation.

In [ ]:
r40 = [json.loads(l) for l in open(os.path.join(RESULTS, "rounds40.jsonl"))]
hu = {(r["rep"], r["seed"]): parse(r["v2"]) for r in map(json.loads, open(os.path.join(RESULTS, "hu_v2.jsonl")))}
rows = []
for r in r40:
    st = [[]]
    for e in r["rounds"]:
        if e["window"] == 15.0 and st[-1]:
            st.append([])
        st[-1].append(e)
    conv = [next((i + 1 for i, e in enumerate(s) if i >= 1 and e["moved_frac"] < 0.01), None) for s in st]
    if r["dataset"] == "c58_8":
        c12 = hu[("c58_8", r["seed"])]
    else:
        c12 = parse(json.load(open(os.path.join(RESULTS, "simulations", f"realistic_seed{r['seed']}_umap-v2.json")))["new"])
    rows.append({"data": r["dataset"], "seed": r["seed"],
                 "stage 1 converged at round": conv[0], "stage 2 converged at round": conv[1] if len(conv) > 1 else None,
                 "median, cap 12": c12[0], "median, cap 40": round(r["final"]["median"], 1),
                 "J@10, cap 12": c12[3], "J@10, cap 40": round(r["final"]["jaccard10"], 3)})
pd.DataFrame(rows).set_index(["data", "seed"])

## 3. Non-uniform bead deposition (Supp Table S5f)

The density prior assumes uniform deposition. Calibrated arrays are simulated with beads deposited according to a smooth log-normal density field (correlation length 150 or 400 µm), at amplitudes spanning about 1 to 13 times the density variation of a uniformly random layout (measured at 300 µm; the experimental replicates span 1.0 to 2.6).

In [ ]:
GRID = 10.0


def density_field(rng, R, ell):
    n = int(np.ceil(2 * R / GRID)) + 1
    g = gaussian_filter(rng.standard_normal((n, n)), ell / GRID / np.sqrt(2), mode="wrap")
    return g / g.std()


def deposit(rng, n, R, g, sigma):
    """n beads in a disc of radius R, density proportional to exp(sigma * g) (rejection sampling)."""
    out, w = [], np.exp(sigma * g)
    while sum(len(o) for o in out) < n:
        r = R * np.sqrt(rng.uniform(0, 1, 4 * n)); t = rng.uniform(0, 2 * np.pi, 4 * n)
        P = np.c_[r * np.cos(t), r * np.sin(t)]
        ij = np.clip(((P + R) / GRID).astype(int), 0, len(g) - 1)
        out.append(P[rng.uniform(0, 1, len(P)) < w[ij[:, 0], ij[:, 1]] / w.max()])
    return np.vstack(out)[:n].astype(np.float32)


def density_relsd(P, center, R, h):
    n = int(np.ceil(2 * R / GRID)); ij = ((P - (center - R)) / GRID).astype(int); ok = ((ij >= 0) & (ij < n)).all(1)
    H = np.zeros((n, n)); np.add.at(H, (ij[ok, 0], ij[ok, 1]), 1.0); S = gaussian_filter(H, h / GRID)
    g = (np.arange(n) + 0.5) * GRID; m = np.hypot(g[:, None] - R, g[None, :] - R) < R - h - 50
    return float(S[m].std() / S[m].mean())


def nonuniformity(P, rng, h=300):
    """Density variation relative to a uniformly random layout of the same beads (1 = as uniform as random)."""
    cen = P.mean(0); R = float(np.percentile(np.linalg.norm(P - cen, axis=1), 85))
    a = rng.uniform(0, 2 * np.pi, len(P)); r = R / 0.85 ** 0.5 * np.sqrt(rng.uniform(0, 1, len(P)))
    return density_relsd(P, cen, R, h) / density_relsd(cen + np.c_[r * np.cos(a), r * np.sin(a)], cen, R, h)


if RECOMPUTE:
    prm = load_params(); R = float(prm["array_diameter"]) / 2
    with open(os.path.join(OUTDIR, "nonuniform_v2.jsonl"), "w") as fh:
        for ell in (150.0, 400.0):
            for sigma in (0.0, 0.03, 0.05, 0.07, 0.1, 0.17, 0.3):
                for s in range(3):
                    rng = np.random.default_rng(1000 + s); g = density_field(rng, R, ell)
                    X = deposit(rng, int(prm["n_receiver"]), R, g, sigma); U = deposit(rng, int(prm["n_sender"]), R, g, sigma)
                    C, Xt, _ = simulate_realistic(seed=s, receivers=X, senders=U); E = umap_embed(C, s)
                    r = {"ell": ell, "sigma": sigma, "seed": s, "nonunif_300": round(nonuniformity(np.vstack([X, U]), np.random.default_rng(s)), 2),
                         "umap": fmt(evaluate(E, Xt)), "accumap_v2": fmt(evaluate(reconstruct(C, E, 2 * R, verbose=False)["receivers"], Xt))}
                    fh.write(json.dumps(r) + "\n"); print(r, flush=True)

nu = [json.loads(l) for l in open(os.path.join(OUTDIR, "nonuniform_v2.jsonl"))]
rows = []
for ell in (150.0, 400.0):
    for sigma in sorted({r["sigma"] for r in nu}):
        G = [r for r in nu if r["ell"] == ell and r["sigma"] == sigma]
        U = np.array([parse(r["umap"]) for r in G]); A = np.array([parse(r["accumap_v2"]) for r in G])
        rows.append({"patch size (um)": int(ell), "non-uniformity": np.mean([r["nonunif_300"] for r in G]),
                     "UMAP median": U[:, 0].mean(), "accumap median": A[:, 0].mean(), "accumap better (of 3)": int((A[:, 0] < U[:, 0]).sum()),
                     "UMAP J@10": U[:, 3].mean(), "accumap J@10": A[:, 3].mean()})
pd.DataFrame(rows).round(3)

## 4. Sensitivity analyses of the single-stage method (Supp Table S5a-e)

These analyses were run during development with the single-stage method (the first stage alone, before the contact-isotropy prior was added): (a) the likelihood temper φ on all replicates, (b) a tolerant density prior, (c) refinement started from the imaging positions, (d) subsampled data, and (e) the length of the kernel fit. They are shown from cached results; their conclusions are discussed in Supplementary Note 2.

In [ ]:
s5 = pd.read_csv(os.path.join(RESULTS, "table_s5_single_stage.csv"))
s5.set_index(["panel", "replicate", "setting"]).drop(columns="panel_title")